# Lesson 3B: DBSCAN - Practical Applications

## Crime Hot Spot Analysis 🚔

Apply DBSCAN to geographic crime data to identify patrol zones.

**Real-world use**: Police departments use density-based clustering to optimize patrols and resource allocation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
# Generate crime hotspot data
hotspots = [
    np.random.randn(150, 2) * 0.3 + [37.7749, -122.4194],
    np.random.randn(100, 2) * 0.4 + [37.7599, -122.4148],
    np.random.randn(200, 2) * 0.15 + [37.7844, -122.4145],
]
X_crime = np.vstack(hotspots)
noise = np.random.randn(50, 2) * 1.5 + [37.76, -122.44]
X_crime = np.vstack([X_crime, noise])

# Visualize
plt.figure(figsize=(10, 8))
plt.scatter(X_crime[:, 1], X_crime[:, 0], alpha=0.5, s=30, edgecolors='k')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Crime Incidents in San Francisco (Simulated)', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()
print(f"✅ {len(X_crime)} crime incidents generated")

In [ ]:
# Standardize and find epsilon with k-distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_crime)

min_pts = 5
nbrs = NearestNeighbors(n_neighbors=min_pts).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances = np.sort(distances[:, min_pts-1])

plt.figure(figsize=(10, 5))
plt.plot(k_distances, linewidth=2)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_pts-1}-Distance')
plt.title('K-Distance Plot for Epsilon Selection', fontweight='bold')
eps_suggest = k_distances[int(len(k_distances)*0.95)]
plt.axhline(y=eps_suggest, color='red', linestyle='--', label=f'ε={eps_suggest:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"✅ Suggested ε={eps_suggest:.3f}")

In [ ]:
# Apply DBSCAN
dbscan = DBSCAN(eps=eps_suggest, min_samples=min_pts)
labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"Clusters: {n_clusters}, Noise: {n_noise}")

# Visualize
plt.figure(figsize=(12, 9))
unique_labels = set(labels)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'black'
        marker = 'x'
    else:
        marker = 'o'
    mask = labels == label
    plt.scatter(X_crime[mask, 1], X_crime[mask, 0], c=[color], marker=marker,
               s=50, alpha=0.7, edgecolors='k', linewidth=0.5,
               label=f'Cluster {label}' if label != -1 else 'Noise')

plt.xlabel('Longitude', fontweight='bold')
plt.ylabel('Latitude', fontweight='bold')
plt.title(f'DBSCAN: {n_clusters} Crime Hot Spots Detected', fontweight='bold')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.show()
print("✅ Hot spots identified!")

In [ ]:
# Compare with K-Means
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# K-Means
ax = axes[0]
scatter = ax.scatter(X_crime[:, 1], X_crime[:, 0], c=kmeans_labels, cmap='viridis',
                    s=50, alpha=0.6, edgecolors='k')
centers = scaler.inverse_transform(kmeans.cluster_centers_)
ax.scatter(centers[:, 1], centers[:, 0], c='red', s=300, marker='*', edgecolors='black', linewidths=2)
ax.set_title('K-Means (k=3)', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

# DBSCAN
ax = axes[1]
for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'black'
        marker = 'x'
    else:
        marker = 'o'
    mask = labels == label
    ax.scatter(X_crime[mask, 1], X_crime[mask, 0], c=[color], marker=marker,
              s=50, alpha=0.6, edgecolors='k')
ax.set_title(f'DBSCAN ({n_clusters} clusters)', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

plt.suptitle('K-Means vs DBSCAN: Crime Hot Spot Detection', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print("✅ DBSCAN finds irregular shapes, K-Means forces spherical clusters")